# LLM Matcher Sanity Check

The LLM step (Gemini) returned 0 matches out of 854 unmatched plants. This notebook verifies the LLM matcher actually works by testing it on **known correct matches** — NPP plants that already matched via rapidfuzz.

- If the LLM **can** match these easy cases → 0/854 is just the expected result for hard cases
- If the LLM **can't** → the prompt or pipeline has a bug

## 1. Setup

In [ ]:
import sys
sys.path.insert(0, "..")

import pandas as pd
from pydantic_settings import BaseSettings

from src.plant_name_matchers.gemini import GeminiNameMatcher
from src.plant_name_matchers.retriever import CandidateRetriever
from src.build_crosswalk import load_gem, load_gppd, SOURCE_COUNTRIES

class Settings(BaseSettings):
    gemini_api_key: str
    model_config = {"env_file": "../.env", "extra": "ignore"}

settings = Settings()
matcher = GeminiNameMatcher(api_key=settings.gemini_api_key)
print(f"Gemini matcher ready (model: {matcher.model})")

## 2. Load test cases — NPP plants that matched via rapidfuzz (known ground truth)

In [ ]:
crosswalk = pd.read_parquet("../data/crosswalks/unified_plant_crosswalk.parquet")

npp_fuzz = crosswalk[
    (crosswalk["source_system"] == "NPP") & (crosswalk["matching_method"] == "rapidfuzz")
].copy()

print(f"NPP rapidfuzz matches available: {len(npp_fuzz)}")

sample = npp_fuzz.sample(5, random_state=42).reset_index(drop=True)
sample[["plant_name", "ref_matched_name", "ref_source"]]

## 3. Build candidate retriever (same as the real pipeline)

In [ ]:
cfg = SOURCE_COUNTRIES["NPP"]

# Load GEM names for India
gem_data = load_gem("NPP")
gem_names = list(gem_data.keys())

# Load GPPD names for India
gppd_countries = [cfg["gppd"]] if cfg.get("gppd") else cfg.get("gppd_countries")
gppd_df = load_gppd(gppd_countries)
gppd_names = gppd_df["name"].dropna().unique().tolist()

print(f"GEM candidates: {len(gem_names)}, GPPD candidates: {len(gppd_names)}")

retriever = CandidateRetriever({"GEM": gem_names, "GPPD": gppd_names})

## 4. Run LLM matcher on each sampled plant

In [ ]:
results = []
candidate_strings = {}  # store for inspection later

for i, row in sample.iterrows():
    plant_name = row["plant_name"]
    known_match = row["ref_matched_name"]
    known_source = row["ref_source"]

    # Get candidates (same as real pipeline)
    candidates_str = retriever.get_candidates(plant_name, limit=15)
    candidate_strings[plant_name] = candidates_str

    # Call LLM
    result = matcher.match(plant_name, candidates_str, source_system="NPP")

    # The LLM returns match with source prefix like "GEM: Agartala power plant"
    # Strip the source prefix for comparison against ref_matched_name
    llm_match_name = result.match
    if llm_match_name and ": " in llm_match_name:
        llm_match_name = llm_match_name.split(": ", 1)[1]

    passed = (
        llm_match_name is not None
        and llm_match_name.strip().lower() == known_match.strip().lower()
    )

    results.append({
        "plant_name": plant_name,
        "rapidfuzz_match": f"{known_source}: {known_match}",
        "llm_match_raw": result.match,
        "llm_match_name": llm_match_name,
        "llm_confidence": result.confidence,
        "llm_source": result.source,
        "pass": passed,
        "reasoning": result.reasoning,
    })

    status = "PASS" if passed else "FAIL"
    print(f"[{i+1}/5] {status} — {plant_name}")
    print(f"         rapidfuzz: {known_match}")
    print(f"         LLM raw:   {result.match}")
    print(f"         reasoning: {result.reasoning}")
    print()

print("Done!")

## 5. Results summary

In [ ]:
results_df = pd.DataFrame(results)

pass_rate = results_df["pass"].sum() / len(results_df)
print(f"Pass rate: {results_df['pass'].sum()}/{len(results_df)} ({pass_rate:.0%})\n")

results_df[["plant_name", "rapidfuzz_match", "llm_match_raw", "llm_confidence", "pass"]]

## 6. Inspect raw prompts (first 3 examples)

Show the exact candidate strings sent to the LLM so we can verify the correct match is present in the candidate list.

In [ ]:
for i, row in sample.head(3).iterrows():
    name = row["plant_name"]
    known = row["ref_matched_name"]
    cands = candidate_strings[name]

    present = any(known.lower() in line.lower() for line in cands.split("\n"))

    print(f"{'='*60}")
    print(f"Plant: {name}")
    print(f"Known match: {known}")
    print(f"Known match in candidates: {'YES' if present else 'NO'}")
    print(f"Candidates sent to LLM:")
    print(cands)
    print()